In [1]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [2]:
from pathlib import Path

DOCS_DIR = Path("docs")

documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}")

Загружено документов: 61


In [3]:
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

Всего чанков: 543
Средняя длина чанка: 543 символов


In [4]:
for chunk in chunks[:3]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

--- из docs\api\anthropic-compatibility.mdx (226 символов) ---
---
title: Anthropic compatibility
---

Connect Anthropic clients and tools such as Claude Code to Ollama. Ollama supports a subset of the [Anthropic Messages API](https://platform.claude.com/docs/en/ ...

--- из docs\api\anthropic-compatibility.mdx (704 символов) ---
## Direct cloud access

Set your [API key](https://ollama.com/settings/keys) in `OLLAMA_API_KEY`. No Ollama installation required.

```shell
export OLLAMA_API_KEY="your_api_key"
```

Send a request wi ...

--- из docs\api\anthropic-compatibility.mdx (153 символов) ---
Compatible models support messages, streaming, and function calling. Tool-choice controls, deferred tools, and hosted web search are not fully supported. ...

